In [2]:
# ruff: noqa: E402, E701
import importlib.util
import json
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import torch
from google.colab import drive, files
from sklearn.decomposition import PCA

drive.mount("/content/drive")
subprocess.run(["pip", "install", "torch", "pyarrow", "catboost", "scikit-learn", "-q"])

print("GPU available:", torch.cuda.is_available())
print(
    "Device:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch to GPU!)",
)

Mounted at /content/drive
GPU available: True
Device: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
FOLDER = "stat_ml_a2"
DRIVE_PATH = f"/content/drive/MyDrive/{FOLDER}"
sys.path.insert(0, DRIVE_PATH)

from tuning import tune_hyperparameters
from metrics import accuracy_score, precision_recall_f1, roc_auc_score, mean_and_error_bar
from models import FTTransformerModel

required = [
    "airbnb_features_train.parquet",
    "airbnb_features_test.parquet",
    "airbnb_features_generalization.parquet",
    "airbnb_sbert_embeddings_train.parquet",
    "airbnb_sbert_embeddings_test.parquet",
    "airbnb_sbert_embeddings_generalization.parquet",
    "airbnb_feature_groups.json",
    "cv.py",
    "tuning.py",
    "metrics.py",
    "models.py",
]
all_ok = True
for f in required:
    path = os.path.join(DRIVE_PATH, f)
    status = "OK" if os.path.exists(path) else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  {status}  {f}")

if not all_ok:
    raise FileNotFoundError("Some files are missing. Check FOLDER path above.")
print("\nAll files found.")

  OK  airbnb_features_train.parquet
  OK  airbnb_features_test.parquet
  OK  airbnb_features_generalization.parquet
  OK  airbnb_sbert_embeddings_train.parquet
  OK  airbnb_sbert_embeddings_test.parquet
  OK  airbnb_sbert_embeddings_generalization.parquet
  OK  airbnb_feature_groups.json
  OK  cv.py
  OK  tuning.py
  OK  metrics.py
  OK  models.py

All files found.


In [ ]:
RAW = "target_unavailable_rate_90"


def filter_active_short_term(frame):
    mask = pd.Series(True, index=frame.index)
    if "minimum_nights" in frame.columns:
        mask &= frame["minimum_nights"] <= 90
    if "has_availability" in frame.columns:
        mask &= frame["has_availability"].isin([True, "t", 1])
    if "number_of_reviews" in frame.columns:
        mask &= frame["number_of_reviews"] > 0
    return frame[mask].reset_index(drop=True)


def merge_embeddings(frame, split):
    emb = pd.read_parquet(os.path.join(DRIVE_PATH, f"airbnb_sbert_embeddings_{split}.parquet"))
    return frame.merge(
        emb, on=["city", "snapshot", "listing_id"], how="left", validate="one_to_one"
    )


def add_embedding_pca(train_frame, test_frame, gen_frame, raw_prefix, out_prefix, n_components=16):
    raw_cols = [c for c in train_frame.columns if c.startswith(raw_prefix)]
    if not raw_cols:
        raise ValueError(f"No SBERT embedding columns found for prefix {raw_prefix}")
    medians = train_frame[raw_cols].median(axis=0)
    pca = PCA(n_components=n_components, svd_solver="randomized", random_state=42)
    pca.fit(train_frame[raw_cols].fillna(medians).to_numpy())
    out_cols = [f"{out_prefix}_pc_{i + 1:02d}" for i in range(n_components)]
    for frame in [train_frame, test_frame, gen_frame]:
        transformed = pca.transform(frame[raw_cols].fillna(medians).to_numpy())
        for i, col in enumerate(out_cols):
            frame[col] = transformed[:, i]
    explained = float(pca.explained_variance_ratio_.sum())
    print(
        f"{out_prefix}: {len(raw_cols)} raw columns -> {len(out_cols)} PCA features; explained_var={explained:.3f}"
    )
    return out_cols


def numeric_cols(frame, cols):
    return [c for c in cols if c in frame.columns and pd.api.types.is_numeric_dtype(frame[c])]

In [5]:
df = filter_active_short_term(
    pd.read_parquet(os.path.join(DRIVE_PATH, "airbnb_features_train.parquet"))
)
df_test = filter_active_short_term(
    pd.read_parquet(os.path.join(DRIVE_PATH, "airbnb_features_test.parquet"))
)
df_gen = filter_active_short_term(
    pd.read_parquet(os.path.join(DRIVE_PATH, "airbnb_features_generalization.parquet"))
)

df = merge_embeddings(df, "train")
df_test = merge_embeddings(df_test, "test")
df_gen = merge_embeddings(df_gen, "generalization")
print(f"Train: {len(df):,} rows x {len(df.columns)} columns")
print(f"Temporal test: {len(df_test):,} rows x {len(df_test.columns)} columns")
print(f"Geographic generalisation: {len(df_gen):,} rows x {len(df_gen.columns)} columns")

host_embedding = add_embedding_pca(
    df, df_test, df_gen, "host_about_sbert_", "host_sbert", n_components=16
)
review_embedding = add_embedding_pca(
    df, df_test, df_gen, "reviews_sbert_", "review_sbert", n_components=16
)
embedding = host_embedding + review_embedding

threshold = float(df[RAW].quantile(0.75))
for frame, name in [(df, "train"), (df_test, "temporal_test"), (df_gen, "generalisation")]:
    frame["target_high_demand"] = (frame[RAW] >= threshold).astype(int)
    print(
        f"{name}: threshold={threshold:.4f}  positive_rate={frame['target_high_demand'].mean():.3f}"
    )
y = df["target_high_demand"].to_numpy()
y_test = df_test["target_high_demand"].to_numpy()
y_gen = df_gen["target_high_demand"].to_numpy()

with open(os.path.join(DRIVE_PATH, "airbnb_feature_groups.json")) as f:
    groups = json.load(f)

base = numeric_cols(df, groups.get("base_controls", []))
text = (
    numeric_cols(df, groups.get("host_nlp", groups.get("host_all", [])))
    + numeric_cols(df, groups.get("review_nlp", groups.get("review_all", [])))
    + numeric_cols(df, groups.get("description_nlp", []))
)

feature_sets = {
    "base_only": base,
    "base_plus_text": base + text,
    "base_plus_text_embedding": base + text + embedding,
}
for name, cols in feature_sets.items():
    missing_test = [c for c in cols if c not in df_test.columns or c not in df_gen.columns]
    print(f"  {name}: {len(cols)} columns; missing in held-out splits: {len(missing_test)}")

Train: 53,348 rows x 875 columns
Temporal test: 52,006 rows x 875 columns
Geographic generalisation: 31,675 rows x 875 columns
host_sbert: 384 raw columns -> 16 PCA features; explained_var=0.663
review_sbert: 384 raw columns -> 16 PCA features; explained_var=0.394
train: threshold=0.7889  positive_rate=0.250
temporal_test: threshold=0.7889  positive_rate=0.253
generalisation: threshold=0.7889  positive_rate=0.227
  base_only: 64 columns; missing in held-out splits: 0
  base_plus_text: 97 columns; missing in held-out splits: 0
  base_plus_text_embedding: 129 columns; missing in held-out splits: 0


In [6]:
def load_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    assert spec is not None
    assert spec.loader is not None
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


cv_mod = load_module("project_cv", os.path.join(DRIVE_PATH, "cv.py"))
k_fold_indices = cv_mod.k_fold_indices
print("All modules imported OK")

All modules imported OK


In [ ]:
OUTER_K = 10
INNER_K = 3
FTTRANSFORMER_GRID = {
    "n_blocks": [4, 6, 8],
    "lr": [1e-4, 3e-4, 1e-3],
    "dropout": [0.1],
}


class MedianImputer:
    def fit_transform(self, X):
        X = np.asarray(X, dtype=float)
        self._meds = np.nanmedian(X, axis=0)
        return self._fill(X.copy())

    def transform(self, X):
        return self._fill(np.asarray(X, dtype=float).copy())

    def _fill(self, X):
        for j in range(X.shape[1]):
            m = np.isnan(X[:, j])
            if m.any():
                X[m, j] = self._meds[j]
        return X


def score_fn(est, Xtr, ytr, Xv, yv):
    return float(precision_recall_f1(yv, est.predict(Xv))["macro_f1"])

In [7]:
outer_splits = list(k_fold_indices(len(df), k=OUTER_K, random_state=42))
all_records = []

for feat_name, feat_cols in feature_sets.items():
    X = df[feat_cols].to_numpy()
    print(f"\n{'=' * 60}")
    print(f"FTTransformer | {feat_name} | {len(feat_cols)} features")
    print(f"{'=' * 60}")

    for fold_i, (tr_idx, te_idx) in enumerate(outer_splits):
        t0 = time.perf_counter()

        imp = MedianImputer()
        X_tr = imp.fit_transform(X[tr_idx])
        X_te = imp.transform(X[te_idx])
        y_tr, y_te = y[tr_idx], y[te_idx]

        inner_splits = list(k_fold_indices(len(X_tr), k=INNER_K, random_state=fold_i * 17))

        best_params, _ = tune_hyperparameters(
            X_tr,
            y_tr,
            estimator_factory=lambda p: FTTransformerModel(batch_size=8192, **p),
            param_grid=FTTRANSFORMER_GRID,
            inner_splits=inner_splits,
            score_function=score_fn,
        )

        model = FTTransformerModel(batch_size=8192, **best_params)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_te)
        scores = model.predict_scores(X_te)
        acc = accuracy_score(y_te, y_pred)
        prf = precision_recall_f1(y_te, y_pred)
        auc = roc_auc_score(y_te, scores)
        elapsed = time.perf_counter() - t0

        record = {
            "model": "FTTransformer",
            "features": feat_name,
            "split": "cv_outer",
            "fold": fold_i + 1,
            "best_params": json.dumps(best_params),
            "accuracy": round(acc, 6),
            "macro_f1": round(float(prf["macro_f1"]), 6),
            "macro_precision": round(float(prf["macro_precision"]), 6),
            "macro_recall": round(float(prf["macro_recall"]), 6),
            "auc": round(auc, 6),
            "elapsed_s": round(elapsed, 1),
        }
        all_records.append(record)
        pd.DataFrame(all_records).to_csv(
            "/content/fttransformer_blocks_lr_sweep_fold_scores.csv", index=False
        )

        print(
            f"  fold {fold_i + 1:2d}  acc={acc:.3f}  f1={float(prf['macro_f1']):.3f}  "
            f"auc={auc:.3f}  {elapsed:.0f}s  best={best_params}"
        )

print("\nAll folds complete!")


FTTransformer | base_only | 64 features
  fold  1  acc=0.880  f1=0.811  auc=0.839  239s  best={'n_blocks': 6, 'lr': 0.001, 'dropout': 0.1}
  fold  2  acc=0.877  f1=0.811  auc=0.839  235s  best={'n_blocks': 6, 'lr': 0.001, 'dropout': 0.1}
  fold  3  acc=0.881  f1=0.814  auc=0.836  230s  best={'n_blocks': 4, 'lr': 0.001, 'dropout': 0.1}
  fold  4  acc=0.867  f1=0.792  auc=0.834  236s  best={'n_blocks': 8, 'lr': 0.001, 'dropout': 0.1}
  fold  5  acc=0.871  f1=0.811  auc=0.842  230s  best={'n_blocks': 6, 'lr': 0.001, 'dropout': 0.1}
  fold  6  acc=0.876  f1=0.808  auc=0.847  228s  best={'n_blocks': 4, 'lr': 0.001, 'dropout': 0.1}
  fold  7  acc=0.878  f1=0.804  auc=0.833  235s  best={'n_blocks': 8, 'lr': 0.001, 'dropout': 0.1}
  fold  8  acc=0.873  f1=0.805  auc=0.842  237s  best={'n_blocks': 8, 'lr': 0.001, 'dropout': 0.1}
  fold  9  acc=0.880  f1=0.804  auc=0.840  232s  best={'n_blocks': 6, 'lr': 0.001, 'dropout': 0.1}
  fold 10  acc=0.875  f1=0.813  auc=0.842  230s  best={'n_blocks': 4

In [ ]:
results_df = pd.DataFrame(all_records)
cv_out = os.path.join(DRIVE_PATH, "fttransformer_blocks_lr_sweep_fold_scores.csv")
results_df.to_csv(cv_out, index=False)
print(f"Saved CV fold scores to Drive: {cv_out}")

print("\n=== FT-Transformer CV Summary (mean ± 95% CI) ===")
for feat_name, g in results_df.groupby("features"):
    print(f"\n  {feat_name}")
    for metric in ["accuracy", "macro_f1", "auc"]:
        mn, ci = mean_and_error_bar(g[metric].to_numpy())
        print(f"    {metric:12s}: {mn:.4f} ± {ci:.4f}")

In [ ]:
def best_params_from_cv(records, feat_name):
    rows = [r for r in records if r["features"] == feat_name and r.get("split") == "cv_outer"]
    score_by_params = {}
    for r in rows:
        key = r["best_params"]
        score_by_params.setdefault(key, []).append(r["macro_f1"])
    best_key, scores = max(score_by_params.items(), key=lambda kv: float(np.mean(kv[1])))
    return json.loads(best_key), float(np.mean(scores))


class MedianImputerFull:
    def fit_transform(self, X):
        X = np.asarray(X, dtype=float)
        self.medians_ = np.nanmedian(X, axis=0)
        return self.transform(X)

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            m = np.isnan(X[:, j])
            if m.any():
                X[m, j] = self.medians_[j]
        return X


def evaluate_split(model, X, y, split):
    y_pred = model.predict(X)
    scores = model.predict_scores(X)
    prf = precision_recall_f1(y, y_pred)
    return {
        "split": split,
        "accuracy": round(accuracy_score(y, y_pred), 6),
        "macro_f1": round(float(prf["macro_f1"]), 6),
        "macro_precision": round(float(prf["macro_precision"]), 6),
        "macro_recall": round(float(prf["macro_recall"]), 6),
        "auc": round(roc_auc_score(y, scores), 6),
    }

In [8]:
model_dir = os.path.join(DRIVE_PATH, "fttransformer_models")
os.makedirs(model_dir, exist_ok=True)
test_records = []

for feat_name, feat_cols in feature_sets.items():
    best_params, cv_macro_f1 = best_params_from_cv(all_records, feat_name)
    print(f"\nFinal fit for {feat_name}: best_params={best_params}, cv_macro_f1={cv_macro_f1:.4f}")

    imp = MedianImputerFull()
    X_train = imp.fit_transform(df[feat_cols].to_numpy())
    X_test = imp.transform(df_test[feat_cols].to_numpy())
    X_gen = imp.transform(df_gen[feat_cols].to_numpy())

    model = FTTransformerModel(batch_size=8192, **best_params)
    model.fit(X_train, y)

    for split, X_eval, y_eval in [
        ("temporal_test", X_test, y_test),
        ("generalisation", X_gen, y_gen),
    ]:
        rec = evaluate_split(model, X_eval, y_eval, split)
        rec.update(
            {
                "model": "FTTransformer",
                "features": feat_name,
                "best_params": json.dumps(best_params),
            }
        )
        test_records.append(rec)
        print(
            f"  {split:16s}: acc={rec['accuracy']:.3f}  f1={rec['macro_f1']:.3f}  auc={rec['auc']:.3f}"
        )

    ckpt = {
        "model": "FTTransformer",
        "feature_set": feat_name,
        "feature_columns": feat_cols,
        "best_params": best_params,
        "target_threshold": threshold,
        "imputer_medians": imp.medians_,
        "scaler_mean": model._scaler.mean_,
        "scaler_scale": model._scaler.scale_,
        "n_features": len(feat_cols),
        "model_config": {
            "n_blocks": model.n_blocks,
            "d_token": model.d_token,
            "n_heads": model.n_heads,
            "ffn_dim": model.ffn_dim,
            "dropout": model.dropout,
            "lr": model.lr,
            "max_epochs": model.max_epochs,
            "patience": model.patience,
            "batch_size": model.batch_size,
        },
        "state_dict": {k: v.detach().cpu() for k, v in model._net.state_dict().items()},
    }
    ckpt_path = os.path.join(model_dir, f"fttransformer_{feat_name}_final.pt")
    torch.save(ckpt, ckpt_path)
    print(f"  saved model checkpoint: {ckpt_path}")

test_df = pd.DataFrame(test_records)
test_out = os.path.join(DRIVE_PATH, "fttransformer_blocks_lr_sweep_test_results.csv")
test_df.to_csv(test_out, index=False)
print(f"\nSaved held-out test results to Drive: {test_out}")

files.download(cv_out)
files.download(test_out)
print("\nDownloads started — check your browser downloads.")

Saved CV fold scores to Drive: /content/drive/MyDrive/stat_ml_a2/fttransformer_blocks_lr_sweep_fold_scores.csv

=== FT-Transformer CV Summary (mean ± 95% CI) ===

  base_only
    accuracy    : 0.8757 ± 0.0027
    macro_f1    : 0.8073 ± 0.0040
    auc         : 0.8393 ± 0.0025

  base_plus_text
    accuracy    : 0.8761 ± 0.0027
    macro_f1    : 0.8074 ± 0.0045
    auc         : 0.8378 ± 0.0037

  base_plus_text_embedding
    accuracy    : 0.8756 ± 0.0024
    macro_f1    : 0.8073 ± 0.0041
    auc         : 0.8342 ± 0.0039

Final fit for base_only: best_params={'n_blocks': 4, 'lr': 0.001, 'dropout': 0.1}, cv_macro_f1=0.8118
  temporal_test   : acc=0.677  f1=0.641  auc=0.784
  generalisation  : acc=0.646  f1=0.607  auc=0.747
  saved model checkpoint: /content/drive/MyDrive/stat_ml_a2/fttransformer_models/fttransformer_base_only_final.pt

Final fit for base_plus_text: best_params={'n_blocks': 8, 'lr': 0.001, 'dropout': 0.1}, cv_macro_f1=0.8144
  temporal_test   : acc=0.719  f1=0.672  auc=0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloads started — check your browser downloads.
